# 07 — Security Properties and Limitations

This scheme is a **reversible encryption scheme** providing semantic security for stored geographic coordinates. It is **not** a formal privacy primitive, not a differential privacy system, and not a substitute for k-anonymity.

This notebook provides an honest inventory of what the scheme does and does not protect, together with concrete directions for improvement.

In [1]:
import secrets
import math
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from map_encryption import (
    MapEncryption, SchemeParams, SCHEME_VERSION,
    _project, _unproject, _R_EARTH,
)

CENTER_LAT, CENTER_LON = 51.513341, -0.136668  # Broadwick Street pump, Soho, London (1854 cholera outbreak)

## Security Property Inventory

The module docstring lists six security properties. Here is what each prevents:

| # | Property | Mechanism | What it prevents |
|---|----------|-----------|------------------|
| 1 | Unbiased PRP via rejection sampling | BLAKE2s with discard loop | Frequency-fingerprint attack on encrypted tile distribution |
| 2 | Length-prefixed Associated Data | `struct.pack('>iiI', qx, qy, len(tweak))` | Boundary-shift: two distinct ADs cannot produce identical bytes |
| 3 | Per-record jitter from nonce | `BLAKE2s(jitter_key, pack(qxp,qyp) + nonce)` | Co-location fingerprinting: same tile, different display pin |
| 4 | Record-ID binding via tweak | `make_tweak(record_id, extra)` | Record-substitution: ciphertext cannot be re-used for another record |
| 5 | Single master key with HKDF-style KDF | BLAKE2s with label + version | Cross-primitive key reuse enabling chosen-plaintext cross-key attacks |
| 6 | Scheme version field | `version = SCHEME_VERSION` in every record | Silent cross-version decryption after key rotation |

In [2]:
principals = [
    'Database reader (no key)',
    'Network observer',
    'Display-tier operator (jitter_key only)',
    'Decode-tier operator (prp_key + aead_key)',
    'Key holder (master key)',
]
can_observe = [
    'qxp, qyp, nonce, ct_resid, tweak, version (all fields)',
    'Record structure in transit',
    'Approximate display position (±62.5 m of shuffled tile)',
    'Exact (lat, lon) of any record',
    'All subkeys; can decrypt anything',
]
cannot_observe = [
    'Original tile (qx, qy), residual (rx, ry), exact location',
    'Plaintext if TLS in use; key material never in transit',
    'Exact GPS coordinates; tile identity; record linkage',
    'Cannot derive master key from subkeys (one-way KDF)',
    'N/A — full access',
]
guarantees = [
    'IND-CPA: records are computationally indistinguishable from random',
    'AEAD integrity: tampering is detected; no location leaks via metadata',
    'Display tier needs only jitter_key; blast radius limited',
    'Full precision recovery; integrity verification on every decode',
    'KMS/HSM custody recommended; audit log all key uses',
]
row_colors = [
    ['lightyellow'] * 5,
    ['lightyellow'] * 5,
    ['lightyellow'] * 5,
    ['lightyellow'] * 5,
    ['lightyellow'] * 5,
]

fig = go.Figure(go.Table(
    header=dict(
        values=['Principal', 'Can observe', 'Cannot observe', 'Scheme guarantee'],
        fill_color='paleturquoise',
        align='left', font=dict(size=11)
    ),
    cells=dict(
        values=[principals, can_observe, cannot_observe, guarantees],
        align='left', font=dict(size=10),
        fill_color=row_colors
    )
))
fig.update_layout(
    title='Threat model: what each principal can and cannot see',
    height=400
)
fig.show()

## Limitation 1 — No Formal Anonymity

This scheme does **not** provide differential privacy or k-anonymity. The encrypted tile indices `(qxp, qyp)` are deterministic for a given tile and key — an attacker who can issue repeated bounding-box queries against `(qxp, qyp)` and observe which records fall within the box may infer statistical areas of interest across millions of records.

For formal location privacy, consider **geo-indistinguishability** (Andres et al., 2013), which adds calibrated Laplace noise to satisfy a differential-privacy guarantee for geographic data.

## Limitation 2 — Access-Pattern Leakage

Encrypting coordinates does **not** hide:
- Which records are fetched from the database
- Query frequency and timing patterns
- Temporal correlations (e.g., a record fetched repeatedly at commute times)

An attacker with database query logs can perform traffic analysis. Mitigations include Oblivious RAM (ORAM) for access-pattern hiding, query-rate limiting, and differential privacy on aggregate query counts.

## Limitation 3 — Web Mercator Distortion

A 250 m Mercator tile covers `250 / cos(lat)` metres of real ground. At 60°N (Oslo, Stockholm, Helsinki), the effective tile size is approximately 500 m — twice the stated bin size. At 80°N (northern Svalbard), a tile spans about 1440 m.

For polar deployments, replace Web Mercator with a local **UTM zone** or **national grid** (e.g., ETRS89-NTM for Norway) which provides near-equal-area tiling in the region of interest.

In [3]:
lat_samples = [0, 20, 40, 60, 80]
effective_m = [250 / math.cos(math.radians(lat)) for lat in lat_samples]

fig = px.bar(
    x=[f'{lat}\u00b0' for lat in lat_samples],
    y=effective_m,
    labels={'x': 'Latitude', 'y': 'Effective ground distance (m)'},
    title='Effective ground distance of a 250 m Mercator tile by latitude',
    template='plotly_white',
    height=400,
    color=effective_m,
    color_continuous_scale='Reds'
)
fig.add_hline(y=250, line_dash='dash', line_color='steelblue',
              annotation_text='250 m reference', annotation_position='top right')
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

## Limitation 4 — Key Compromise is Catastrophic

Master key compromise exposes all three subkeys simultaneously, allowing full decryption of the entire database. There is no forward secrecy.

Production mitigations:
- Store master key in a **KMS or HSM** (AWS KMS, GCP Cloud KMS, HashiCorp Vault)
- **Key rotation**: bump `SCHEME_VERSION`, re-encrypt with new key, retire old key after all records are migrated
- **Audit logging**: log every key usage (subkey derivation, decode call)
- **Per-component access controls**: display tier receives only `jitter_key`

## Limitation 5 — Bespoke Construction

This scheme assembles well-studied standard primitives (BLAKE2s, ChaCha20-Poly1305) in a **custom** Feistel construction. Custom constructions, however carefully designed, lack the extensive peer review of published standards.

For production use:
- Obtain an **external cryptographic audit** of the full scheme
- Consider replacing the custom Feistel PRP with **NIST-standardised FF3-1** (NIST SP 800-38G Rev. 1), which is a Format-Preserving Encryption scheme designed exactly for this kind of finite-domain permutation use case

## Directions for Improvement

1. **Replace the Feistel PRP with FF3-1** (NIST SP 800-38G Rev. 1) for a standardised, audited format-preserving permutation over the tile index domain.

2. **Store keys in KMS/HSM** with fine-grained IAM policies, audit logs, and automatic rotation schedules.

3. **Add differential privacy for data releases**: when publishing aggregate statistics, add calibrated Laplace or Gaussian noise to satisfy ε-geo-indistinguishability.

4. **Use UTM for polar deployments** (lat > 60°): UTM zones provide near-equal-area tiling and eliminate the Mercator scale distortion problem.

5. **Formalise record schema with authenticated version negotiation**: embed a HMAC over the version field so downgrade attacks (forging `version=1` on a v2 record) are detected.

## References

- **Snow, J.** (1855). *On the Mode of Communication of Cholera* (2nd ed.). Churchill, London. — Source of the 1854 Soho cholera death and pump location dataset used throughout these notebooks.
- **Lin, J.** (2023). Geo-indistinguishable masking: enhancing privacy protection in spatial point mapping. See `docs/Geo-indistinguishablemaskingenhancingprivacyprotectioninspatialpointmapping.pdf`.
- **Luby, M., & Rackoff, C.** (1988). How to construct pseudorandom permutations from pseudorandom functions. *SIAM Journal on Computing, 17*(2), 373–386. — Proof that ≥ 4 Feistel rounds produce a PRP.
- **Nir, Y., & Langley, A.** (2018). ChaCha20 and Poly1305 for IETF Protocols. RFC 8439. IETF. — Specification for the AEAD cipher used in this library.
- **NIST SP 800-38G Rev. 1** (2024). *Recommendation for Block Cipher Modes of Operation: Methods for Format-Preserving Encryption.* NIST. — Standard for FF3-1, the recommended PRP replacement cited in NB07.
- **Dwork, C.** (2006). Differential privacy. *Proceedings of ICALP 2006*, LNCS 4052, 1–12. — Context for Limitation 1 (no formal anonymity) in NB07.
- **Geoprivacy Ethics** (2024). *Ethical tensions in public health geoprivacy.* Internal framework document; see `docs/Geoprivacy ethics.pdf`.